In [1]:
!pip install h5py

Looking in links: /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/x86-64-v3, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/generic, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/generic
ERROR: Could not find a version that satisfies the requirement h5py (from versions: none)
ERROR: No matching distribution found for h5py


In [9]:
import pickle
import os
import h5py
import gc

In [10]:
def get_model_name(job_id: int):
    id_to_model_name = {
        0: "alexnet",
        1: "resnet50",
        2: "resnet101",
        3: "vgg19",
        4: "vit_b_16",
        5: "vit_b_32",
        6: "efficientnet_v2_s",
        7: "resnet18",
        8: "inception_v3",
    }

    return id_to_model_name[job_id]

def get_it_layer(model_name: str):
    it_layer_dict = {
        "alexnet": "features.12",
        "resnet50": "layer3.2.bn1",
        "resnet101": "layer3.2.bn1",
        "vgg16": "features.30",
        "vgg19": "features.36",
        "inception_v3": "Mixed_7a.branch3x3_1.bn",
        "vit_b_16": "encoder.layers.encoder_layer_8.mlp",
        "vit_b_32": "encoder.layers.encoder_layer_8.mlp",
        "efficientnet_v2_s": "features.6.7.stochastic_depth",
        "resnet18": "layer4.0.relu",
    }
    return it_layer_dict[model_name]

In [11]:
feature_path = "/home/soroush1/projects/def-kohitij/soroush1/training_fast_publish_faster/pool_layers_pkl"


tasks = "no_model"

for i in range(9):

    print("-"*100)
    model_idx = i
    model_name = get_model_name(model_idx)
    it_layer = get_it_layer(model_name)

    print(f"{model_name} - {tasks}: {it_layer}")
    
    with open(os.path.join(feature_path, f"{model_name}_{tasks}_1.pkl"), "rb") as fin:
        features = pickle.load(fin)
    
    print(f"{features.keys()}\n is in? {it_layer in features.keys()}")

    del features
    gc.collect()

----------------------------------------------------------------------------------------------------
alexnet - no_model: features.12
dict_keys(['x', 'features.1', 'features.3', 'features.6', 'features.8', 'features.11', 'features.12', 'avgpool', 'classifier.1', 'classifier.3', 'classifier.6'])
 is in? True
----------------------------------------------------------------------------------------------------
resnet50 - no_model: layer3.2.bn1
dict_keys(['maxpool', 'layer1.0.add', 'layer1.2.add', 'layer2.0.add', 'layer2.2.add', 'layer3.0.downsample.0', 'layer3.1.add', 'layer3.2.bn1', 'layer3.3.add', 'layer3.5.add', 'layer4.0.add'])
 is in? True
----------------------------------------------------------------------------------------------------
resnet101 - no_model: layer3.2.bn1
dict_keys(['maxpool', 'layer1.1.add', 'layer2.0.add', 'layer2.3.add', 'layer3.1.add', 'layer3.2.bn1', 'layer3.4.add', 'layer3.7.add', 'layer3.10.add', 'layer3.13.add', 'layer3.16.add'])
 is in? True
-----------------

In [ ]:
# write a function to take, feature_path, and model_name, and task_name, and save .h5 file
# for that model and task
# then call this function for all models and tasks

def save_h5(feature_path, model_name, task_name, layer_name):
    with open(os.path.join(feature_path, f"{model_name}_{task_name}.pkl"), "rb") as fin:
        features = pickle.load(fin)
        
    data = features[layer_name]
    
    # Save the NumPy array as a .h5 file
    with h5py.File(f'{model_name}_{task_name}.h5', 'w') as hf:
        hf.create_dataset('dataset_name', data=data)
        
    